## Imports & setting up:

In [ ]:
import os

os.environ["PYKEOPS_VERBOSE"] = "0"
os.environ["KEOPS_VERBOSE"] = "0"

In [ ]:
import sys
sys.path.append(os.path.join(os.getcwd(), '../../')) # Add root of repo to import MBM

import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from cmcrameri import cm
import massbalancemachine as mbm
import logging
import torch 
from matplotlib.colors import LinearSegmentedColormap
import matplotlib as mpl
from matplotlib.ticker import FuncFormatter
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colorbar as mcolorbar

from matplotlib.lines import Line2D
import scipy.stats as scipy_stats
from regions.TF_Europe.scripts.config_TF_Europe import *
from regions.TF_Europe.scripts.dataset import *
from regions.TF_Europe.scripts.plotting import *
from regions.TF_Europe.scripts.models import *
from regions.TF_Europe.scripts.models import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

# Initialize logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
BASE_DIR = Path(cfg.dataPath) / path_cache / f"CROSS_REGION"
print(f"Base directory for data: {BASE_DIR}")

MONTHLY_COLS = [
    't2m',
    'tp',
    'slhf',
    'sshf',
    'ssrd',
    'fal',
    'str',
    'ELEVATION_DIFFERENCE',
]
STATIC_COLS = ['aspect', 'slope', 'svf']

feature_columns = MONTHLY_COLS + STATIC_COLS

# --- plot KDEs ---
COL_LABELS = {
    "t2m": "2m temperature",
    "tp": "Precipitation",
    "slhf": "Latent heat flux",
    "sshf": "Sensible heat flux",
    "ssrd": "Solar radiation",
    "fal": "Albedo",
    "str": "Thermal radiation",
    "ELEVATION_DIFFERENCE": "Elevation diff. (m)",
    "aspect": "Aspect (°)",
    "slope": "Slope (°)",
    "svf": "Sky-view factor",
}

# Transform data to monthly format (run or load data):
paths = {
    'era5_climate_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_monthly_averaged_data_EU_US_CANADA.nc"),
    'geopotential_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_geopotential_pressure_EU_US_CANADA.nc")
}

# Check that all these files exists
for key, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file for {key} not found at {path}")

nature_seq = LinearSegmentedColormap.from_list(
    "nature_seq",
    [
        NATURE_PALETTE["sky_blue"], NATURE_PALETTE["blue"],
        NATURE_PALETTE["black"]
    ],
)


## Load glacier grids:

In [ ]:
"""
Examples of loading data:
# Load Switzerland only
df = load_stakes(cfg, "CH")

# Load all Central Europe (FR+CH+IT+AT when you add them)
df_ceu = load_stakes_for_rgi_region(cfg, "11")

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}"""

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}
dfs["11"].head(2)

### Add subregions:

In [ ]:
# Central Asia
dfs, glacier_dict_ca = add_o2region_to_dfs(dfs, '13', cfg)

# Alaska
dfs, glacier_dict_alaska = add_o2region_to_dfs(dfs,
                                               '01',
                                               cfg,
                                               deduplicate_glaciers=True)

## Maps

In [ ]:
def build_region_glacier_info(
    cfg,
    *,
    rgi_region_id: str,
    outline_shp_path: str,
    glacier_col="GLACIER",
    lat_col="POINT_LAT",
    lon_col="POINT_LON",
    period_col="PERIOD",
    nmeas_col="Nb. measurements",
    source_col="SOURCE_CODE",
    source_resolution="mode",
    o2region_col="O2Region",  # : optional second-level region column
    load_stakes_fn=None,
    verbose=True,
):
    """
    Generic builder for per-glacier info tables (for maps / summaries), for any region.
    No FT/holdout/train-test labeling — just location, measurement counts, source,
    and (if present) O2 sub-region.
    """

    if load_stakes_fn is None:
        load_stakes_fn = load_stakes_for_rgi_region  # noqa: F821

    data_region = load_stakes_fn(cfg, rgi_region_id)

    # --- O2Region split (Central Asia / Alaska only) ---
    if rgi_region_id == "13":
        dfs, glacier_dict = add_o2region_to_dfs({rgi_region_id: data_region},
                                                "13", cfg)
        data_region = dfs[rgi_region_id]
    elif rgi_region_id == "01":
        dfs, glacier_dict = add_o2region_to_dfs({rgi_region_id: data_region},
                                                "01",
                                                cfg,
                                                deduplicate_glaciers=True)
        data_region = dfs[rgi_region_id]

    glacier_outline_rgi = gpd.read_file(outline_shp_path)

    if verbose:
        print(f"[{rgi_region_id}] stake rows: {len(data_region)} | "
              f"glaciers: {data_region[glacier_col].nunique()}")

    # --- measurement counts ---
    meas_counts = (data_region.groupby(glacier_col).size().sort_values(
        ascending=False).rename(nmeas_col).to_frame())

    # --- mean location ---
    glacier_loc = data_region.groupby(glacier_col)[[lat_col, lon_col]].mean()

    # --- counts per period (winter/annual) ---
    glacier_period = (data_region.groupby(
        [glacier_col, period_col]).size().unstack().fillna(0).astype(int))

    # --- SOURCE_CODE per glacier ---
    if source_col in data_region.columns:
        gsrc = data_region.groupby(glacier_col)[source_col].apply(
            lambda s: s.dropna().astype(str).unique())

        mixed = gsrc[gsrc.apply(len) > 1]
        if len(mixed) > 0 and verbose:
            print(
                f"Warning: {len(mixed)} glaciers have multiple {source_col} values "
                f"(showing up to 5): {mixed.head(5).to_dict()}")

        if len(mixed) > 0 and source_resolution == "error":
            raise ValueError(
                f"Found glaciers with multiple {source_col} values. "
                f"Set source_resolution to 'first', 'mode', or 'list' to resolve."
            )

        if source_resolution == "list":
            glacier_source = (gsrc.apply(lambda arr: list(arr)).rename(
                source_col).to_frame())
        elif source_resolution == "first":
            glacier_source = (gsrc.apply(lambda arr: arr[0] if len(arr) else
                                         None).rename(source_col).to_frame())
        elif source_resolution == "mode":

            def _mode(series):
                s = series.dropna().astype(str)
                if len(s) == 0:
                    return None
                return s.value_counts().index[0]

            glacier_source = (data_region.groupby(glacier_col)[source_col].
                              apply(_mode).rename(source_col).to_frame())
        else:
            raise ValueError(
                "source_resolution must be one of: 'error','first','mode','list'"
            )
    else:
        glacier_source = None
        if verbose:
            print(
                f"Note: '{source_col}' not found in data_region; skipping SOURCE_CODE merge."
            )

    # --- O2Region per glacier (mode, in case of any inconsistency) ---
    if o2region_col in data_region.columns:

        def _mode(series):
            s = series.dropna().astype(str)
            if len(s) == 0:
                return None
            return s.value_counts().index[0]

        gregion = data_region.groupby(glacier_col)[o2region_col].apply(_mode)

        n_unique_per_glacier = data_region.groupby(
            glacier_col)[o2region_col].nunique()
        inconsistent = n_unique_per_glacier[n_unique_per_glacier > 1]
        if len(inconsistent) > 0 and verbose:
            print(
                f"Warning: {len(inconsistent)} glaciers have multiple {o2region_col} "
                f"values (using mode): {list(inconsistent.index[:5])}")

        glacier_o2region = gregion.rename(o2region_col).to_frame()
    else:
        glacier_o2region = None
        if verbose:
            print(
                f"Note: '{o2region_col}' not found in data_region; skipping O2Region merge."
            )

    # --- merge base ---
    glacier_info = glacier_loc.join(meas_counts,
                                    how="inner").join(glacier_period,
                                                      how="left")
    if glacier_source is not None:
        glacier_info = glacier_info.join(glacier_source, how="left")
    if glacier_o2region is not None:
        glacier_info = glacier_info.join(glacier_o2region, how="left")

    return data_region, glacier_outline_rgi, glacier_info

In [ ]:
from regions.TF_Europe.scripts.plotting.style import NATURE_COLORS, nature_figsize, NATURE_SPECS


def plot_glacier_measurements_map(
    glacier_info,
    glacier_outline_rgi,
    *,
    lon_col="POINT_LON",
    lat_col="POINT_LAT",
    nmeas_col="Nb. measurements",
    hue_col=None,
    palette=None,  # : dict {category: color}, or None -> NATURE_COLORS
    cmap_for_hue=None,
    title="Glacier measurement locations",
    figsize=None,  # : None -> nature_figsize(cols=1)
    nature_cols=1,  # : 1 or 2, used only if figsize is None
    nature_height_mm=80,  # : used only if figsize is None
    extent=(5.8, 15, 44, 48),
    sizes=(
        20,
        300),  # : smaller default range, more sensible at Nature panel scale
    size_legend_values=(30, 100, 1000, 6000),
    alpha=0.6,
    point_color=NATURE_COLORS[5],  # "blue" from NATURE_PALETTE
    land_facecolor="lightgray",
    land_alpha=0.5,
    add_features=True,
    add_gridlines=True,
    legend_loc="lower right",
    legend_ncol=2,
    legend_fontsize=NATURE_SPECS["font_max_pt"],
    legend_title_fontsize=NATURE_SPECS["font_max_pt"],
    hue_legend_title=None,
    title_fontsize=NATURE_SPECS["font_max_pt"] + 1,
    gridline_label_fontsize=NATURE_SPECS["font_max_pt"] - 1,
    zorder_scatter=10,
    data_region=None,
    year_col="YEAR",
    summary_loc=(0.02, 0.02),
    summary_fontsize=NATURE_SPECS["font_max_pt"],
    show=True,
):
    """
    Plot glacier point locations on a Cartopy map with marker size ~ sqrt(# measurements),
    styled to Nature figure specs (font sizes, dpi, colors via NATURE_COLORS).

    If hue_col is given, points are colored by that categorical column (e.g. O2Region)
    using NATURE_COLORS by default (or a custom palette / cmap_for_hue), and a combined
    hue+size legend is built. Otherwise, single color for all points.

    Requires (in your environment):
      - numpy as np
      - matplotlib.pyplot as plt
      - seaborn as sns
      - cartopy.crs as ccrs
      - cartopy.feature as cfeature
      - from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
      - matplotlib.lines import Line2D
      - utils.plotting.style: NATURE_COLORS, nature_figsize, NATURE_SPECS

    Returns
    -------
    fig, ax : matplotlib Figure and Axes
    glacier_info_out : copy of glacier_info with added ['sqrt_size','scaled_size']
    scaled_size_fn : function(val)->float used for consistent scaling
    """

    if figsize is None:
        figsize = nature_figsize(cols=nature_cols, height_mm=nature_height_mm)

    # --- 0) Copy input ---
    df = glacier_info.copy()

    # --- 1) Preprocessing: sqrt scaling ---
    if nmeas_col not in df.columns:
        raise KeyError(f"Missing '{nmeas_col}' in glacier_info.")
    df["sqrt_size"] = np.sqrt(df[nmeas_col].astype(float))

    sqrt_min = float(df["sqrt_size"].min())
    sqrt_max = float(df["sqrt_size"].max())

    def scaled_size(val, min_out=sizes[0], max_out=sizes[1]):
        sqrt_val = np.sqrt(float(val))
        if sqrt_max == sqrt_min:
            return (min_out + max_out) / 2
        return min_out + (max_out - min_out) * ((sqrt_val - sqrt_min) /
                                                (sqrt_max - sqrt_min))

    df["scaled_size"] = df[nmeas_col].apply(scaled_size)

    # --- 1b) Hue setup ---
    use_hue = hue_col is not None
    if use_hue:
        if hue_col not in df.columns:
            raise KeyError(f"Missing '{hue_col}' in glacier_info.")
        df[hue_col] = df[hue_col].astype(str)
        categories = sorted(df[hue_col].dropna().unique())

        if palette is None:
            if cmap_for_hue is not None:
                try:
                    colors = get_cmap_hex(cmap_for_hue,
                                          len(categories))  # noqa: F821
                except Exception:
                    colors = NATURE_COLORS[:len(categories)]
            else:
                if len(categories) > len(NATURE_COLORS):
                    print(
                        f"Warning: {len(categories)} categories but only "
                        f"{len(NATURE_COLORS)} NATURE_COLORS available; colors will repeat."
                    )
                colors = [
                    NATURE_COLORS[i % len(NATURE_COLORS)]
                    for i in range(len(categories))
                ]
            palette = dict(zip(categories, colors))
        if hue_legend_title is None:
            hue_legend_title = hue_col

    # --- 2) Figure + map ---
    fig = plt.figure(figsize=figsize)

    lonW, lonE, latS, latN = extent

    max_lat, min_lat = float(df[lat_col].max()), float(df[lat_col].min())
    max_lon, min_lon = float(df[lon_col].max()), float(df[lon_col].min())

    if not (latS <= min_lat <= max_lat <= latN):
        print(
            f"Warning: latitude bounds may not fully contain glacier points: "
            f"glacier lat range [{min_lat:.2f}, {max_lat:.2f}] vs map bounds [{latS}, {latN}]"
        )
    if not (lonW <= min_lon <= max_lon <= lonE):
        print(
            f"Warning: longitude bounds may not fully contain glacier points: "
            f"glacier lon range [{min_lon:.2f}, {max_lon:.2f}] vs map bounds [{lonW}, {lonE}]"
        )

    projPC = ccrs.PlateCarree()
    ax = plt.axes(projection=projPC)
    ax.set_extent([lonW, lonE, latS, latN], crs=ccrs.Geodetic())

    if add_features:
        ax.add_feature(cfeature.COASTLINE, linewidth=0.4)
        ax.add_feature(cfeature.LAKES, linewidth=0.3)
        ax.add_feature(cfeature.RIVERS, linewidth=0.3)
        ax.add_feature(cfeature.BORDERS, linestyle="-", linewidth=0.4)
        ax.add_feature(cfeature.LAND,
                       facecolor=land_facecolor,
                       alpha=land_alpha)

    glacier_outline_rgi.plot(ax=ax,
                             transform=projPC,
                             color="black",
                             alpha=0.7,
                             linewidth=0.4)

    # --- 3) Scatter ---
    if use_hue:
        g = sns.scatterplot(
            data=df,
            x=lon_col,
            y=lat_col,
            size="scaled_size",
            hue=hue_col,
            sizes=sizes,
            alpha=alpha,
            palette=palette,
            transform=projPC,
            ax=ax,
            zorder=zorder_scatter,
            legend=True,
        )
    else:
        g = sns.scatterplot(
            data=df,
            x=lon_col,
            y=lat_col,
            size="scaled_size",
            sizes=sizes,
            alpha=alpha,
            color=point_color,
            transform=projPC,
            ax=ax,
            zorder=zorder_scatter,
            legend=False,
        )

    # --- 4) Gridlines ---
    if add_gridlines:
        gl = ax.gridlines(
            draw_labels=True,
            linewidth=0.3,
            color="gray",
            alpha=0.5,
            linestyle="--",
        )
        gl.xformatter = LONGITUDE_FORMATTER
        gl.yformatter = LATITUDE_FORMATTER
        gl.xlabel_style = {"size": gridline_label_fontsize, "color": "black"}
        gl.ylabel_style = {"size": gridline_label_fontsize, "color": "black"}
        gl.top_labels = gl.right_labels = False

    # --- 5) Legend ---
    size_handles = [
        Line2D(
            [],
            [],
            marker="o",
            linestyle="None",
            markersize=np.sqrt(scaled_size(val)),
            markerfacecolor=(point_color if not use_hue else "gray"),
            alpha=alpha,
            label=f"{val}",
        ) for val in size_legend_values
    ]

    if use_hue:
        handles, labels = g.get_legend_handles_labels()
        expected_labels = list(palette.keys())
        label_to_handle = dict(zip(labels, handles))
        hue_entries = [(label_to_handle[l], l) for l in expected_labels
                       if l in label_to_handle]

        combined_handles = [h for h, _ in hue_entries] + size_handles
        combined_labels = [l for _, l in hue_entries
                           ] + [str(v) for v in size_legend_values]
        legend_title = f"{hue_legend_title} / Number of measurements"
    else:
        combined_handles = size_handles
        combined_labels = [str(v) for v in size_legend_values]
        legend_title = "Number of measurements"

    leg = ax.legend(
        combined_handles,
        combined_labels,
        title=legend_title,
        loc=legend_loc,
        frameon=True,
        fontsize=legend_fontsize,
        title_fontsize=legend_title_fontsize,
        borderpad=0.8,
        labelspacing=0.8,
        ncol=legend_ncol,
    )
    leg.get_frame().set_linewidth(0.4)

    # --- 6) Summary text box ---
    if data_region is not None:
        n_glaciers = len(glacier_info.index.unique())

        if year_col in data_region.columns:
            years = data_region[year_col].dropna()
            if len(years) > 0:
                y_min, y_max = int(years.min()), int(years.max())
                year_str = f"{y_min}\u2013{y_max}" if y_min != y_max else f"{y_min}"
            else:
                year_str = "N/A"
        else:
            print(
                f"Note: '{year_col}' not found in data_region; skipping years in summary."
            )
            year_str = "N/A"

        summary_text = f"Glaciers: {n_glaciers}\nYears: {year_str}"

        ax.text(
            summary_loc[0],
            summary_loc[1],
            summary_text,
            transform=ax.transAxes,
            fontsize=summary_fontsize,
            verticalalignment="bottom",
            horizontalalignment="left",
            bbox=dict(boxstyle="round",
                      facecolor="white",
                      alpha=0.8,
                      edgecolor="gray",
                      linewidth=0.4),
            zorder=zorder_scatter + 1,
        )

    ax.set_title(title, fontsize=title_fontsize)
    plt.tight_layout()

    if show:
        plt.show()

    return fig, ax, df, scaled_size

### CEU:

In [ ]:
data_CEU, glacier_outline_rgi, glacier_df_CEU = build_region_glacier_info(
    cfg,
    rgi_region_id="11",
    outline_shp_path=cfg.dataPath +
    "RGI_v6/RGI_11_CentralEurope/11_rgi60_CentralEurope.shp",
)

fig, ax, glacier_info_plot, scaled_size_fn = plot_glacier_measurements_map(
    glacier_info=glacier_df_CEU,
    data_region=data_CEU,
    glacier_outline_rgi=glacier_outline_rgi,
    title="Glacier measurement locations Central European Alps",
    extent=(5.8, 15, 44, 48),
    sizes=(50, 800),
    size_legend_values=(30, 100, 1000),
    figsize=(10, 8))

# save figure
fig.savefig('figures/paperTF/fig_maps_CEU_gl.pdf')

### NOR:

In [ ]:
data_NOR, glacier_outline_rgi, glacier_df_NOR = build_region_glacier_info(
    cfg,
    rgi_region_id="08",
    outline_shp_path=cfg.dataPath +
    "RGI_v6/RGI_08_Scandinavia/08_rgi60_Scandinavia.shp",
)

fig, ax, glacier_info_plot, scaled_size_fn = plot_glacier_measurements_map(
    glacier_info=glacier_df_NOR,
    data_region=data_NOR,
    glacier_outline_rgi=glacier_outline_rgi,
    title="Glacier measurement locations Norway",
    extent=(4, 24, 57, 71),
    sizes=(50, 800),
    size_legend_values=(30, 100, 1000),
    figsize=(15, 10))

# save figure
fig.savefig('figures/paperTF/fig_maps_NOR_gl.pdf')

### Iceland:

In [ ]:
data_ISL, glacier_outline_rgi, glacier_df_ISL = build_region_glacier_info(
    cfg,
    rgi_region_id="06",
    outline_shp_path=cfg.dataPath +
    "RGI_v6/RGI_06_Iceland/06_rgi60_Iceland.shp",
)

fig, ax, glacier_info_plot, scaled_size_fn = plot_glacier_measurements_map(
    glacier_info=glacier_df_ISL,
    data_region=data_ISL,
    glacier_outline_rgi=glacier_outline_rgi,
    title="Glacier measurement locations Iceland",
    extent=(-25, -12, 63, 67),
    sizes=(50, 800),
    size_legend_values=(30, 100, 1000),
    figsize=(10, 8))

# save figure
fig.savefig('figures/paperTF/fig_maps_ISL_gl.pdf')

### Svalbard:

In [ ]:
data_SJM, glacier_outline_rgi, glacier_df_SJM = build_region_glacier_info(
    cfg,
    rgi_region_id="07",
    outline_shp_path=cfg.dataPath +
    "RGI_v6/RGI_07_Svalbard/07_rgi60_Svalbard.shp",
)

fig, ax, glacier_info_plot, scaled_size_fn = plot_glacier_measurements_map(
    glacier_info=glacier_df_SJM,
    data_region=data_SJM,
    glacier_outline_rgi=glacier_outline_rgi,
    title="Glacier measurement locations Svalbard",
    extent=(10, 25, 76.5, 80),
    sizes=(300, 800),
    size_legend_values=(30, 50, 200),
    figsize=(15, 10))

# save figure
fig.savefig('figures/paperTF/fig_maps_SJM_gl.pdf')

### Alaska:

In [ ]:
data_ALA, glacier_outline_rgi, glacier_df_ALA = build_region_glacier_info(
    cfg,
    rgi_region_id="01",
    outline_shp_path=cfg.dataPath + "RGI_v6/RGI_01_Alaska/01_rgi60_Alaska.shp",
)

palette_ala = {
    "2": NATURE_PALETTE["orange"],
    "4": NATURE_PALETTE["blue"],
    "6": NATURE_PALETTE["bluish_green"],
}

fig, ax, glacier_info_plot, scaled_size_fn = plot_glacier_measurements_map(
    glacier_info=glacier_df_ALA,
    data_region=data_ALA,
    glacier_outline_rgi=glacier_outline_rgi,
    title="Glacier measurement locations Alaska",
    extent=(-170, -130, 55, 67),
    sizes=(50, 800),
    size_legend_values=(5, 50, 500),
    figsize=(10, 8),
    hue_col='O2Region',
    legend_loc='upper right',
    palette=palette_ala)

# save figure
fig.savefig('figures/paperTF/fig_maps_ALA_gl.pdf')

### Central Asia:

In [ ]:
data_CA, glacier_outline_rgi, glacier_df_CA = build_region_glacier_info(
    cfg,
    rgi_region_id="13",
    outline_shp_path=cfg.dataPath +
    "RGI_v6/RGI_13_CentralAsia/13_rgi60_CentralAsia.shp",
)

palette_ca = {
    "1": NATURE_PALETTE["orange"],
    "2": NATURE_PALETTE["blue"],
    "3": NATURE_PALETTE["bluish_green"],
    "4": NATURE_PALETTE["reddish_purple"],
}

fig, ax, glacier_info_plot, scaled_size_fn = plot_glacier_measurements_map(
    glacier_info=glacier_df_CA,
    data_region=data_CA,
    glacier_outline_rgi=glacier_outline_rgi,
    title="Glacier measurement locations Central Asia",
    extent=(67, 90, 35, 46),
    sizes=(100, 1000),
    size_legend_values=(30, 500, 1000),
    figsize=(10, 8),
    hue_col='O2Region',
    legend_loc='lower right',
    palette=palette_ca)

# save figure
fig.savefig('figures/paperTF/fig_maps_CA_gl.pdf')